# 07MIAR — Redes Neuronales y Deep Learning
## Proyecto de programación: *Deep Vision in classification tasks*

**Actividad:** Comparación de estrategias para la clasificación de imágenes

**Autor(es):** Adelso Steve Araya Solórzano - Fabio Antonio Rodriguez Quesada - Bhuvana Cheemala

**Dataset:** Human Action Recognition (HAR) Dataset   https://www.kaggle.com/datasets/meetnagadia/human-action-recognition-har-dataset

**Problema de clasificación:** clasificación multiclase de imágenes fijas en color según la acción humana que representan (15 clases).

## Índice

1. Planteamiento del problema y selección del dataset
2. Carga e inspección de los datos
3. Acondicionamiento y preprocesamiento
4. Estrategia 1 - CNN desde cero (*from scratch*)
5. Estrategia 2 - Modelos preentrenados (*transfer learning* y *fine-tuning*)
6. Evaluación y comparación global
7. Conclusiones
8. Referencias
9. Lista de comprobación y exportación a PDF

**Requisito central:** comparar una CNN entrenada desde cero con, al menos, dos arquitecturas preentrenadas sobre ImageNet. Justificar experimentalmente las decisiones y evaluar finalmente sobre datos no utilizados durante el entrenamiento.

# 1. Planteamiento del problema y selección del dataset

## 1.1 Descripción del problema

**Problema de clasificación:** El problema que se presenta es la identificación de 15 clases de acciones humanas (Human Action Recognition) a partir de imágenes fijas, donde cada imagen muestra a una o varias personas realizando una actividad cotidiana.

**Aplicación o motivación:** La clasificación de imágenes mediante técnicas de Deep Learning representa una herramienta de gran potencial para automatizar tareas de reconocimiento y análisis visual. La motivación de este proyecto es explorar cómo diferentes arquitecturas de redes neuronales pueden aprender a identificar patrones y clasificar imágenes, comparando el rendimiento de modelos construidos desde cero con modelos preentrenados mediante Transfer Learning. Con ello, se busca comprender las ventajas y limitaciones de cada enfoque, así como evaluar cómo influyen las distintas técnicas de entrenamiento y optimización en la precisión y capacidad de generalización de los modelos.

**Variable objetivo y significado de las clases:** Cada clase identifica un tipo de actividad humana. Las clases que contiene el dataset son:

| Clase | Descripción |
|-------|-------------|
| 0     | calling     |
| 1     | clapping    |
| 2     | cycling     |
| 3     | dancing     |
| 4     | drinking    |
| 5     | eating      |
| 6     | fighting    |
| 7     | hugging     |
| 8     | laughing    |
| 9     | listening_to_music     |
| 10    | running     |
| 11    | sitting     |
| 12    | sleeping    |
| 13    | texting     |
| 14    | using_laptop|


## 1.2 Dataset seleccionado

- **Características principales:** conjunto de fotografías reales en color (no sintéticas ni en escala de grises) que muestran a una o varias personas realizando 15 actividades cotidianas distintas, en escenarios, fondos, poses e iluminación muy variables. Procede de una competición pública de Kaggle y ya viene dividido en entrenamiento y test.
- Nombre: Human Action Recognition (HAR)
- Fuente y enlace: (https://www.kaggle.com/datasets/meetnagadia/human-action-recognition-har-dataset)
- Estructura de directorios y etiquetas: **TEST:** Dataset/HumanActionRecognition/Testing_set.csv  **TRAINING:** Dataset/HumanActionRecognition/Training_set.csv . Adicionalmente, existen dos carpetas con las imagenes individuales de cada clase en el directorio Dataset/HumanActionRecognition/train/ y Dataset/HumanActionRecognition/test/ respectivamente.
- Número total de imágenes: 18000 (training data 12600) (test data 5400, sin etiqueta de clase)
- Número de clases: 15 clases
- **Distribución de clases:** el conjunto de entrenamiento está perfectamente balanceado, con exactamente 840 imágenes por clase (15 × 840 = 12600; recuento calculado en la sección 2.3). No existe, por tanto, desequilibrio de clases que deba corregirse.
- **Dimensiones y formatos de imagen:** fotografías JPG en color con dimensiones variables; el ancho, el alto, la relación de aspecto y el modo de color de **todas** las imágenes se calculan en la sección 2.5. Por ello es necesario redimensionar las imágenes a un tamaño común antes de entrenar (128×128 en la estrategia *from scratch* y 224×224 en los modelos preentrenados).

## 1.3 Justificación de la elección

El dataset HAR resulta adecuado para este proyecto por varias razones. En primer lugar, se trata de un problema de clasificación de imágenes con una variable objetivo claramente definida (15 clases de actividades humanas) y con un volumen de datos suficiente (12600 imágenes de entrenamiento) para entrenar y comparar modelos de Deep Learning. En segundo lugar, el conjunto está perfectamente balanceado entre clases, lo que permite evaluar el rendimiento de los modelos sin necesidad de aplicar técnicas de corrección de desbalanceo, aislando así el efecto de las decisiones de arquitectura y entrenamiento. Por último, al tratarse de imágenes reales (no sintéticas) con fondos, poses, iluminación y encuadres variables, el dataset presenta una complejidad visual mayor que otros conjuntos más sencillos (como MNIST o Sign Language MNIST), lo que permite ilustrar de forma más clara las ventajas del *transfer learning* frente a una CNN entrenada desde cero con un número limitado de datos.

In [ ]:
# 1.4 Configuración inicial e importaciones
import os
import logging
import warnings

# Silenciar los mensajes informativos de TensorFlow/absl (deben fijarse ANTES de importar TF)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import Sequential, layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import funciones

tf.get_logger().setLevel(logging.ERROR)
try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
except ImportError:
    pass
warnings.filterwarnings("ignore", category=UserWarning, module=r"keras(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning)

# La inicialización del dispositivo GPU (Metal) escribe mensajes informativos de C++ directamente
# en stderr; se inicializa aquí con stderr redirigido para que no aparezcan en el PDF.
_stderr_fd, _devnull_fd = os.dup(2), os.open(os.devnull, os.O_WRONLY)
os.dup2(_devnull_fd, 2)
try:
    tf.config.list_logical_devices("GPU")
finally:
    os.dup2(_stderr_fd, 2)
    os.close(_devnull_fd)
    os.close(_stderr_fd)

# Semillas para favorecer la reproducibilidad (Python, NumPy y TensorFlow).
# En GPU algunas operaciones no son deterministas: la reproducibilidad es aproximada.
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Parámetros generales del proyecto
IMG_SIZE = (128, 128)   # resolución común de entrada (estrategia from scratch)
BATCH_SIZE = 32         # batch por defecto; cada experimento indica explícitamente el suyo
KAGGLE_DATASET = "meetnagadia/human-action-recognition-har-dataset"
MODELS_DIR = "modelos"  # mejores pesos de cada experimento (carpeta no versionada)
os.makedirs(MODELS_DIR, exist_ok=True)

# Modo de prueba rápida, solo para comprobar que el notebook se ejecuta de principio
# a fin: subconjunto estratificado pequeño y un máximo de 2 épocas por entrenamiento.
# La ejecución entregada se realiza con QUICK_RUN = False (valor por defecto).
QUICK_RUN = os.environ.get("QUICK_RUN", "0") == "1"


def epocas(n):
    """Número máximo de épocas de un entrenamiento (limitado a 2 si QUICK_RUN)."""
    return min(n, 2) if QUICK_RUN else n


print("TensorFlow", tf.__version__, "| GPU disponibles:", tf.config.list_physical_devices("GPU"))
print("QUICK_RUN =", QUICK_RUN)

# 2. Carga e inspección de los datos

## 2.1 Descarga y preparación inicial

El dataset se descarga directamente desde Kaggle mediante la librería `kagglehub` (ver celda siguiente), tal y como recomienda el enunciado, en lugar de subir manualmente los ficheros.

## 2.2 Organización de los datos

El dataset está organizado, dentro de la ruta local devuelta por `kagglehub`, de la siguiente forma:

- `Training_set.csv`: contiene, para cada imagen de entrenamiento, su nombre de fichero (`filename`) y su etiqueta de clase (`label`).
- `Testing_set.csv`: contiene únicamente el nombre de fichero (`filename`) de las imágenes de test; **no incluye la etiqueta de clase**, por lo que no puede utilizarse para calcular métricas de evaluación.
- `train/`: carpeta con las 12600 imágenes de entrenamiento en formato JPG, referenciadas por `filename` en `Training_set.csv`.
- `test/`: carpeta con las 5400 imágenes de test (sin etiqueta) en formato JPG, referenciadas en `Testing_set.csv`.

Dado que el test de Kaggle no está etiquetado, la evaluación final del modelo (sección 6) se realizará sobre una partición de test obtenida internamente a partir del conjunto de entrenamiento etiquetado (división estratificada entrenamiento/validación/test 70/15/15), tal y como se detalla en la sección 3 de acondicionamiento.

In [ ]:
# 2.1 Descarga del dataset (función reutilizable) y verificación de rutas
# pip install kagglehub
import kagglehub

def descargar_dataset_har(handle: str = KAGGLE_DATASET) -> str:
    """
    Descarga el dataset Human Action Recognition desde Kaggle Hub
    (o reutiliza la copia ya cacheada localmente si ya se descargó antes)
    y devuelve la ruta local donde quedan disponibles los CSV y las
    carpetas de imágenes.

    Requiere credenciales de Kaggle configuradas (ver sección 2.1).
    """
    ruta_local = kagglehub.dataset_download(handle)
    print("Dataset disponible en:", ruta_local)
    return ruta_local

# NOTA: kagglehub descomprime el .zip de Kaggle tal cual, y este dataset en
# concreto contiene una carpeta intermedia llamada "Human Action Recognition"
# antes de llegar a los CSV y a las carpetas train/test. Lo comprobamos con
# os.listdir para no depender de un nombre de carpeta fijo.
ruta_descarga = descargar_dataset_har()
subcarpetas = [d for d in os.listdir(ruta_descarga)
               if os.path.isdir(os.path.join(ruta_descarga, d))]
assert len(subcarpetas) == 1, f"Se esperaba una única subcarpeta, se encontraron: {subcarpetas}"
BASE_PATH = os.path.join(ruta_descarga, subcarpetas[0])
print("Carpeta base del dataset:", BASE_PATH)

# Rutas derivadas de la ruta base descargada por kagglehub
TRAIN_CSV = os.path.join(BASE_PATH, "Training_set.csv")
TEST_CSV = os.path.join(BASE_PATH, "Testing_set.csv")
TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR = os.path.join(BASE_PATH, "test")

for ruta in [TRAIN_CSV, TEST_CSV, TRAIN_DIR, TEST_DIR]:
    assert os.path.exists(ruta), f"No se encontró la ruta esperada: {ruta}"

print("Rutas del dataset verificadas correctamente.")

In [ ]:
# 2.2 Lectura de los CSV y tamaño de cada partición original

# Entrenamiento (con etiqueta)
df_train = pd.read_csv(TRAIN_CSV)
X_train = df_train.drop('label', axis=1)
y_train = df_train['label']

# Test (sin etiqueta, proporcionado por Kaggle)
df_test = pd.read_csv(TEST_CSV)
X_test = df_test

print(f"Imágenes de entrenamiento: {X_train.shape[0]}")
print(f"Imágenes de test (sin etiqueta): {X_test.shape[0]}")
print(f"Número de clases en entrenamiento: {y_train.nunique()}")
# El desglose por clase se analiza en detalle en la sección 3.1.

## 2.3 Inspección y análisis exploratorio: distribución de clases

Se cuenta el número de imágenes por clase del conjunto etiquetado y se representa gráficamente para detectar posibles desequilibrios, que obligarían a usar pesos de clase o métricas específicas. La salida de la celda siguiente confirma que el conjunto está balanceado (mismo número de imágenes en las 15 clases), por lo que la *accuracy* es una métrica adecuada y la referencia de un clasificador aleatorio es 1/15 ≈ 0,067.

In [ ]:
# 2.3 Número de imágenes por clase y representación gráfica

conteo_clases = y_train.value_counts().sort_index()
print("Número de imágenes por clase (entrenamiento):")
print(conteo_clases)

df_plot = conteo_clases.reset_index()
df_plot.columns = ["clase", "cantidad"]

plt.figure(figsize=(12, 6))
sns.barplot(
    x="clase",
    y="cantidad",
    hue="clase",
    data=df_plot,
    order=df_plot["clase"],
    palette="viridis",
    legend=False,
)

plt.title(
    "Distribución de la Variable Objetivo en Entrenamiento",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Clase", fontsize=12)
plt.ylabel("Cantidad de Muestras", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## 2.4 Inspección visual de imágenes

Se muestran dos imágenes de ejemplo por cada una de las 15 clases, tal y como se hace en el notebook de referencia de la asignatura. Esto permite detectar visualmente problemas de calidad, variabilidad o posibles errores de etiquetado antes de diseñar el preprocesamiento.

In [ ]:
# Mostrar ejemplos representativos de cada clase para identificar
# posibles problemas de calidad, variabilidad o etiquetado.
# Rejilla compacta 5x6 (dos imágenes por clase) para que la figura quepa en una página del PDF.

clases_ordenadas = sorted(y_train.unique())
n_ejemplos_por_clase = 2
n_columnas = 6

n_imagenes = len(clases_ordenadas) * n_ejemplos_por_clase
n_filas = int(np.ceil(n_imagenes / n_columnas))
fig, axes = plt.subplots(n_filas, n_columnas, figsize=(2.4 * n_columnas, 2.5 * n_filas))
axes = axes.ravel()

k = 0
for clase in clases_ordenadas:
    archivos_clase = X_train.loc[y_train == clase, "filename"].sample(
        n_ejemplos_por_clase, random_state=SEED
    ).values
    for filename in archivos_clase:
        img = cv2.imread(os.path.join(TRAIN_DIR, filename))
        axes[k].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[k].set_title(clase, fontsize=10)
        k += 1
for ax in axes:
    ax.axis("off")

plt.suptitle("Ejemplos representativos por clase", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2.5 Dimensiones y formato de las imágenes

Se leen las cabeceras de **todas** las imágenes (entrenamiento etiquetado y test de Kaggle) para obtener su ancho, alto, relación de aspecto y modo de color, sin decodificarlas por completo. Estas estadísticas justifican el redimensionado a un tamaño fijo de la sección 3 y la elección de las resoluciones de entrada (128×128 en la estrategia *from scratch*, 224×224 en los modelos preentrenados).

**Resultado:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# 2.5 Dimensiones, relación de aspecto y modo de color de todas las imágenes
from PIL import Image


def dimensiones_imagenes(directorio, nombres):
    """Ancho, alto y modo de color leídos de la cabecera de cada imagen."""
    filas = []
    for nombre in nombres:
        with Image.open(os.path.join(directorio, nombre)) as im:
            filas.append({"ancho": im.width, "alto": im.height, "modo": im.mode})
    return pd.DataFrame(filas)


dims = pd.concat([
    dimensiones_imagenes(TRAIN_DIR, X_train["filename"]).assign(particion="train (etiquetado)"),
    dimensiones_imagenes(TEST_DIR, X_test["filename"]).assign(particion="test Kaggle (sin etiqueta)"),
], ignore_index=True)
dims["aspecto (ancho/alto)"] = dims["ancho"] / dims["alto"]

display(dims.groupby("particion")[["ancho", "alto", "aspecto (ancho/alto)"]]
        .describe().T.round(2))
print("Modos de color:", dims.groupby("particion")["modo"].value_counts().to_dict())
print("Resoluciones distintas en train:", dims[dims["particion"].str.startswith("train")]
      .groupby(["ancho", "alto"]).ngroups)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
dims_tr = dims[dims["particion"].str.startswith("train")]
for ax, col in zip(axes, ["ancho", "alto", "aspecto (ancho/alto)"]):
    ax.hist(dims_tr[col], bins=40, color=funciones.COLOR_TRAIN)
    ax.set_title(f"Distribución de {col} (train)", fontsize=11, weight="bold")
    ax.set_xlabel(col if col.startswith("aspecto") else f"{col} (px)"); ax.set_ylabel("Nº de imágenes")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# 3. Acondicionamiento y preprocesamiento

Las decisiones de preprocesamiento para la estrategia *from scratch* son las siguientes:

- **Lectura y color:** OpenCV lee las imágenes en BGR; se convierten a RGB, el orden que esperan las redes preentrenadas de la sección 5 y la visualización.
- **Redimensionado a 128×128:** las imágenes tienen tamaños variables (sección 2.5) y una CNN necesita una entrada de tamaño fijo. 128×128 es un compromiso entre conservar el detalle de la pose y de los objetos (teléfono, portátil, bicicleta…) y el coste de entrenar desde cero; con 12600 imágenes en `float32` el conjunto ocupa ≈ 2,5 GB en memoria. No se conserva la relación de aspecto: la ligera deformación se considera aceptable.
- **Normalización a [0, 1]:** se divide entre 255 y se almacena en `float32`. Una escala de entrada homogénea y acotada estabiliza el descenso del gradiente. Los modelos preentrenados (sección 5) aplicarán en su lugar el `preprocess_input` de cada arquitectura a partir de los ficheros originales.
- **Codificación de etiquetas:** `LabelEncoder` asigna un entero 0–14 a cada clase (orden alfabético), compatible con `sparse_categorical_crossentropy`.
- **División estratificada 70/15/15:** como el test de Kaggle no tiene etiquetas, el conjunto etiquetado se divide en entrenamiento, validación y test interno, conservando la proporción de clases y con semilla fija. **La validación es el único conjunto usado para monitorizar, aplicar *early stopping* y seleccionar modelos en todos los experimentos; el test interno no se utiliza hasta la evaluación final (sección 6).** Se guardan los índices de cada partición (`idx_tr`, `idx_val`, `idx_test`) para que la sección 5 reconstruya exactamente la misma división a partir de las rutas de los ficheros.
- **Data augmentation:** no se aplica aquí, sino mediante capas de Keras dentro del modelo (activas solo durante el entrenamiento), y se evalúa experimentalmente en la sección 4.4.

In [ ]:
# 3.1 Lectura, conversión a RGB y redimensionado de las imágenes etiquetadas
if QUICK_RUN:
    # Solo para la prueba rápida: subconjunto estratificado de 60 imágenes por clase
    df_train = (df_train.groupby("label", group_keys=False)
                .sample(n=60, random_state=SEED)
                .sort_index().reset_index(drop=True))

X_imagenes_uint8, fallidas = funciones.cargar_imagenes(TRAIN_DIR, df_train["filename"], IMG_SIZE)

# Si alguna imagen no se puede leer, se elimina de df_train ANTES de dividir,
# de modo que los índices de las particiones sigan siendo coherentes con df_train.
if fallidas:
    print(f"Se descartan {len(fallidas)} imágenes ilegibles:",
          df_train.loc[fallidas, "filename"].tolist())
    df_train = df_train.drop(index=fallidas).reset_index(drop=True)
assert len(X_imagenes_uint8) == len(df_train)

print("Imágenes cargadas:", X_imagenes_uint8.shape, X_imagenes_uint8.dtype)

In [ ]:
# 3.2 Codificación de las etiquetas a valores numéricos
encoder = LabelEncoder()
y_codificadas = encoder.fit_transform(df_train["label"])
class_names = [str(c) for c in encoder.classes_]
NUM_CLASSES = len(class_names)

print("Número de clases:", NUM_CLASSES)
for indice, nombre_clase in enumerate(class_names):
    print(f"{indice}: {nombre_clase}")

In [ ]:
# 3.3 División estratificada entrenamiento / validación / test interno (70/15/15)
indices = np.arange(len(df_train))
idx_tr, idx_temp = train_test_split(
    indices, test_size=0.30, random_state=SEED, stratify=y_codificadas
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=SEED, stratify=y_codificadas[idx_temp]
)
assert len(set(idx_tr) & set(idx_val)) == 0 and len(set(idx_tr) & set(idx_test)) == 0 \
    and len(set(idx_val) & set(idx_test)) == 0


def preparar_particion(idx):
    """Selecciona una partición y normaliza los píxeles a [0, 1] en float32."""
    return X_imagenes_uint8[idx].astype("float32") / 255.0, y_codificadas[idx]


X_tr, y_tr = preparar_particion(idx_tr)
X_val, y_val = preparar_particion(idx_val)
X_test_interno, y_test_interno = preparar_particion(idx_test)  # reservado para la sección 6
del X_imagenes_uint8  # libera memoria

# Alias usados en la sección 4: solo la partición de entrenamiento (70 %)
X_train_final, y_train_numerico = X_tr, y_tr

print("Entrenamiento:", X_tr.shape, "| Validación:", X_val.shape, "| Test interno:", X_test_interno.shape)
print(f"Rango de píxeles tras normalizar: [{X_tr.min():.1f}, {X_tr.max():.1f}]")

# Comprobación de la estratificación: imágenes por clase en cada partición
reparto = pd.DataFrame({
    "entrenamiento": np.bincount(y_tr, minlength=NUM_CLASSES),
    "validación": np.bincount(y_val, minlength=NUM_CLASSES),
    "test interno": np.bincount(y_test_interno, minlength=NUM_CLASSES),
}, index=class_names)
display(reparto)

# 4. Estrategia 1 — CNN desde cero (*from scratch*)

## 4.1 Diseño de la arquitectura inicial

La red recibe las imágenes de 128×128×3 normalizadas en la sección 3. Como punto de partida para la estrategia *from scratch* se propone una arquitectura CNN mínima, sobre la que se irán aplicando mejoras de forma progresiva en la sección 4.4:

- **Bloques convolucionales:** 2 bloques, cada uno formado por una capa `Conv2D` seguida de `MaxPooling2D`. Dos bloques son suficientes para un primer modelo de referencia (*baseline*) sin ser excesivamente profundos, lo que permite entrenar rápido y medir una primera línea base antes de añadir complejidad.
- **Número de filtros:** 32 filtros en el primer bloque y 64 en el segundo, siguiendo el patrón habitual de duplicar el número de filtros a medida que se reduce la resolución espacial (patrón visto en el notebook de referencia de la asignatura).
- **Tamaño de kernel:** 3×3 en ambas capas convolucionales, el tamaño estándar que ofrece un buen equilibrio entre capacidad de detección de patrones locales y número de parámetros.
- **Función de activación:** ReLU en las capas convolucionales, por ser la opción estándar que evita el problema de desvanecimiento del gradiente propio de activaciones saturantes (sigmoid, tanh).
- **Pooling:** `MaxPooling2D` de 2×2 tras cada bloque convolucional, para reducir la dimensionalidad espacial y aportar cierta invariancia a pequeñas traslaciones.
- **Capas finales:** `Flatten` seguido directamente de una capa `Dense` con activación `softmax` y una neurona por clase (15). No se incluye, de momento, una capa densa oculta intermedia, para mantener el modelo inicial lo más simple posible; en la sección 4.4 se evalúa experimentalmente si añadir una capa densa oculta (con regularización) mejora el rendimiento.

Esta arquitectura se trata como un modelo de referencia (*baseline*) deliberadamente simple. Las siguientes secciones (4.4) documentan los experimentos realizados a partir de este punto de partida (dropout, regularización L2, batch normalization, data augmentation, aumento de filtros/capas, etc.), justificando cada cambio con los resultados obtenidos.

In [ ]:
# 4.1 Arquitectura CNN desde cero (baseline inicial)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

cnn_HAR = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="baseline")

cnn_HAR.summary()

## 4.2 Configuración del entrenamiento

- Optimizador: Adam
- Learning rate: 0.001 (valor por defecto de Adam)
- Batch size: 128
- Número de épocas: 15
- Función de pérdida: `sparse_categorical_crossentropy`
- Métricas: accuracy
- Callbacks: ninguno que modifique el entrenamiento en este modelo baseline (ni *EarlyStopping* ni *scheduler*; se introducen en la sección 4.4)

Adam se elige por ser una opción estándar que combina *momentum* y adaptación automática de la tasa de aprendizaje, lo que favorece una convergencia estable sin ajuste manual en este primer modelo. Como las etiquetas están codificadas como enteros (`LabelEncoder`, no *one-hot*), la pérdida adecuada es `sparse_categorical_crossentropy`. De forma intencionada no se aplica ninguna regularización, para observar el comportamiento de referencia, incluido un posible sobreajuste.

**Procedimiento común a todos los experimentos.** Todos los entrenamientos de la sección 4 se lanzan con `funciones.entrenar_experimento`, que compila el modelo, entrena validando **siempre sobre el mismo conjunto `X_val`/`y_val`** (sección 3), mide el tiempo, imprime un resumen (mejor `val_accuracy` y su época, mínima `val_loss`, parámetros y tiempo), dibuja las curvas y añade una fila al registro `resultados`. Además guarda en `modelos/<experimento>.keras` los pesos de la época con mayor `val_accuracy` (un `ModelCheckpoint` que no altera el entrenamiento). Los historiales se conservan en `historiales[<experimento>]`. Se usa `verbose=2` (una línea por época).

In [ ]:
# 4.2 Entrenamiento del baseline
# Registro común de resultados (métricas de validación) e historiales de todos los experimentos
resultados = []
historiales = {}

history_cnn_HAR = funciones.entrenar_experimento(
    "exp00_baseline", cnn_HAR, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(15), batch_size=128, optimizer=Adam(learning_rate=1e-3),
    callbacks=None, resultados=resultados, historiales=historiales,
    models_dir=MODELS_DIR, mostrar_curvas=False,
)

## 4.3 Monitorización del entrenamiento

La celda siguiente representa la evolución de la *accuracy* y de la pérdida en entrenamiento y validación del baseline (la línea punteada marca la época de mayor `val_accuracy`).

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación:** [PENDIENTE: analizar a partir de las curvas si existe sobreajuste (divergencia entre la pérdida de entrenamiento y la de validación) y en qué época se produce la mejor generalización].

Este modelo se entrena deliberadamente sin ninguna técnica de regularización (ver sección 4.2), con el objetivo de establecer un comportamiento de referencia que motive los experimentos de la sección 4.4 (*dropout*, regularización L2, *early stopping*, *data augmentation* y *batch normalization*).

In [ ]:
# 4.3 Curvas de entrenamiento y validación del baseline
h = historiales["exp00_baseline"]
funciones.impresion(range(1, len(h["accuracy"]) + 1), h, titulo="exp00_baseline")

## 4.4 Experimentación progresiva y regularización

A partir del baseline se realiza una secuencia de experimentos en la que cada uno introduce un cambio concreto, motivado por el resultado del anterior. Condiciones comunes para que los resultados sean comparables:

- Mismas particiones: entrenamiento `X_train_final` (70 %) y validación `X_val` (15 %); el test interno no se usa en esta sección.
- Semilla fija (`tf.keras.utils.set_random_seed(SEED)`) antes de construir cada modelo.
- `EarlyStopping(monitor="val_loss", restore_best_weights=True)` desde el experimento 1, con paciencia 4 en los experimentos cortos (máximo 15 épocas) y 8–12 en los largos (40–60 épocas, con *data augmentation*). Cuando se usa `ReduceLROnPlateau`, su paciencia es menor que la del *EarlyStopping* para que llegue a actuar antes de detener el entrenamiento.
- Criterio de comparación: mejor `val_accuracy` (y `val_loss` asociada); el registro completo se resume al final de la sección.

Técnicas de regularización evaluadas: *dropout* y regularización L2 (exp. 1–3), *data augmentation* (exp. 6), *batch normalization* (exp. 7–10), `GlobalAveragePooling2D` como sustituto de `Flatten` (exp. 8–10), `SpatialDropout2D` y *weight decay* (exp. 10).

*Nota:* respecto a una versión previa del notebook se han fusionado dos experimentos redundantes para ajustar el coste computacional total: (i) uno que repetía la arquitectura del experimento 6 cambiando solo los *callbacks*, que dejó de aportar información al corregir el *scheduler* del experimento 6 (su `min_lr` coincidía con el learning rate inicial, por lo que nunca reducía la tasa); y (ii) una variante del experimento 8 que solo modificaba la cabeza clasificadora y el tamaño de batch sobre la misma red convolucional (la más costosa por época de la sección); la simplificación de la cabeza se estudia de forma más radical en el experimento 9, que la elimina por completo.

**Experimento 1: Dropout + capa densa con regularización L2**

**Objetivo e hipótesis:** combatir el sobreajuste del baseline con dos técnicas de regularización: *dropout* (anula aleatoriamente el 30 % de las activaciones durante el entrenamiento, lo que evita la coadaptación de neuronas) y *weight regularization* L2 en una nueva capa densa oculta (penaliza los pesos grandes y favorece soluciones más suaves). Se añade `EarlyStopping` para detener el entrenamiento cuando `val_loss` deja de mejorar y conservar los mejores pesos. Se espera una menor brecha entre entrenamiento y validación y una `val_loss` mínima inferior a la del baseline.

**Cambios respecto al experimento anterior:** + `Dropout(0.3)` tras el segundo bloque convolucional; + `Dense(128, activation="relu", kernel_regularizer=L2(0.01))` antes de la capa de salida; + `EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)`. Se mantienen Adam (lr = 0.001), batch 128 y un máximo de 15 épocas.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 1: Dropout(0.3) + Dense(128) con L2(0.01) + EarlyStopping
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

modelo_exp01 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),                      # nuevo: dropout

    layers.Flatten(),
    layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(0.01)),  # nuevo: capa densa con L2
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp01")
modelo_exp01.summary()

history_exp01 = funciones.entrenar_experimento(
    "exp01_dropout_l2", modelo_exp01, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(15), batch_size=128, optimizer=Adam(learning_rate=1e-3),
    callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 2: reducción del learning rate**

**Objetivo e hipótesis:** estudiar el efecto de la tasa de aprendizaje. Un learning rate menor produce actualizaciones más pequeñas y una convergencia más suave, lo que podría retrasar el sobreajuste y estabilizar la `val_loss`, con el riesgo de que el modelo no llegue a converger en 15 épocas.

**Cambios respecto al experimento anterior:** misma arquitectura, regularización (Dropout 0.3, L2 = 0.01) y `EarlyStopping` que el experimento 1; el único cambio es el learning rate de Adam, de 0.001 a **0.0001**.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 2: misma red que el experimento 1 con Adam(learning_rate=1e-4)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

modelo_exp02 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp02")
modelo_exp02.summary()

history_exp02 = funciones.entrenar_experimento(
    "exp02_lr_1e-4", modelo_exp02, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(15), batch_size=128,
    optimizer=Adam(learning_rate=1e-4),          # cambio: learning rate 1e-3 -> 1e-4
    callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 3: menor intensidad de la regularización L2**

**Objetivo e hipótesis:** comprobar si la penalización L2 = 0.01 es excesiva y limita la capacidad de la capa densa para aprender representaciones útiles. Se espera que una penalización menor mejore el ajuste sin aumentar en exceso el sobreajuste.

**Cambios respecto al experimento anterior:** misma configuración que el experimento 2 (Adam lr = 0.0001, Dropout 0.3, `EarlyStopping`); L2 de la capa densa de **0.01 a 0.004**.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 3: como el experimento 2 con L2 = 0.004 en la capa densa
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

modelo_exp03 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(0.004)),  # cambio: L2 0.01 -> 0.004
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp03")
modelo_exp03.summary()

history_exp03 = funciones.entrenar_experimento(
    "exp03_l2_4e-3", modelo_exp03, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(15), batch_size=128, optimizer=Adam(learning_rate=1e-4),
    callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 4: mayor capacidad del modelo**

**Objetivo e hipótesis:** comprobar si la red de dos bloques es demasiado simple para la complejidad visual de las 15 acciones (subajuste). Más filtros por bloque y un tercer bloque convolucional permiten extraer patrones de mayor nivel (partes del cuerpo, objetos), con un campo receptivo mayor.

**Cambios respecto al experimento anterior:** filtros 32 → 64 y 64 → 128 en los dos primeros bloques; + tercer bloque `Conv2D(256)` + `MaxPooling2D`, seguido de `Dropout(0.3)` (además del dropout del bloque 2). El factor L2 de la capa densa no se fija a mano: el código toma el valor (0.01 o 0.004) con el que se obtuvo mayor `val_accuracy` entre los experimentos 2 y 3, y se mantiene en los experimentos 4–7. Se conservan Adam (lr = 0.0001), batch 128, 15 épocas y `EarlyStopping`.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 4: más filtros y un tercer bloque convolucional
# Factor L2 elegido a partir de los resultados de validación de los experimentos 2 y 3
val_exp = {r["experimento"]: r["best_val_accuracy"] for r in resultados}
L2_DENSA = 0.004 if val_exp["exp03_l2_4e-3"] > val_exp["exp02_lr_1e-4"] else 0.01
print(f"L2 seleccionado para la capa densa: {L2_DENSA} "
      f"(exp02: {val_exp['exp02_lr_1e-4']:.4f} | exp03: {val_exp['exp03_l2_4e-3']:.4f})")

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

modelo_exp04 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(64, (3, 3), activation="relu"),    # 32 -> 64 filtros
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),   # 64 -> 128 filtros
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(256, (3, 3), activation="relu"),   # nuevo tercer bloque
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_DENSA)),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp04")
modelo_exp04.summary()

history_exp04 = funciones.entrenar_experimento(
    "exp04_mas_capacidad", modelo_exp04, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(15), batch_size=128, optimizer=Adam(learning_rate=1e-4),
    callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 5: learning rate dinámico (ReduceLROnPlateau)**

**Objetivo e hipótesis:** combinar las ventajas de ambos learning rates: comenzar con un valor alto (0.001) para avanzar rápido y reducirlo automáticamente cuando la `val_loss` se estanca, para afinar la solución. Se espera alcanzar en 15 épocas una `val_loss` menor que con un learning rate fijo.

**Cambios respecto al experimento anterior:** misma arquitectura que el experimento 4; Adam vuelve a lr = **0.001** y se añade `ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6)`. La paciencia del *scheduler* (2) es menor que la del `EarlyStopping` (4) para que la reducción del learning rate pueda producirse antes de detener el entrenamiento.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 5: arquitectura del experimento 4 + Adam(1e-3) + ReduceLROnPlateau
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

modelo_exp05 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_DENSA)),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp05")
modelo_exp05.summary()

history_exp05 = funciones.entrenar_experimento(
    "exp05_reduce_lr", modelo_exp05, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(15), batch_size=128, optimizer=Adam(learning_rate=1e-3),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6, verbose=1),
    ],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 6: data augmentation**

**Objetivo e hipótesis:** con solo 8820 imágenes de entrenamiento, la red puede memorizar los ejemplos concretos. El *data augmentation* genera en cada época versiones transformadas de las imágenes (volteo horizontal, rotación y zoom leves), transformaciones que no cambian la acción representada, lo que actúa como regularizador y debería reducir la brecha entre entrenamiento y validación. Como el problema de aprendizaje se vuelve más difícil, se permite entrenar más épocas.

**Cambios respecto al experimento anterior:** + bloque de *data augmentation* al inicio del modelo (`RandomFlip("horizontal")`, `RandomRotation(0.1)`, `RandomZoom(0.1)`, activo solo en entrenamiento); máximo de épocas 15 → **50**; `EarlyStopping` con paciencia 10 y `ReduceLROnPlateau(factor=0.2, patience=4, min_lr=1e-6)`. La arquitectura convolucional es la del experimento 5, para aislar el efecto del *data augmentation*.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 6: arquitectura del experimento 5 + data augmentation (hasta 50 épocas)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

data_augmentation = Sequential([   # transformaciones aleatorias, solo activas en entrenamiento
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

modelo_exp06 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation,

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_DENSA)),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp06")
modelo_exp06.summary()

history_exp06 = funciones.entrenar_experimento(
    "exp06_data_augmentation", modelo_exp06, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(50), batch_size=128, optimizer=Adam(learning_rate=1e-3),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=4, min_lr=1e-6, verbose=1),
    ],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 7: Batch Normalization + LeakyReLU**

**Objetivo e hipótesis:** la *batch normalization* normaliza las activaciones de cada capa por *mini-batch*, lo que estabiliza la distribución de entradas de las capas siguientes, permite aprender más rápido y añade un ligero efecto regularizador. LeakyReLU mantiene un gradiente pequeño para entradas negativas y evita neuronas "muertas" que dejan de aprender. Se espera una convergencia más rápida y estable y una mejor `val_accuracy` que en el experimento 6.

**Cambios respecto al experimento anterior:** en los tres bloques convolucionales y en la capa densa oculta, `Conv2D/Dense(activation="relu")` se sustituye por `Conv2D/Dense(use_bias=False)` → `BatchNormalization()` → `LeakyReLU(negative_slope=0.01)` (el sesgo es redundante porque la BN incluye su propio desplazamiento). Se mantienen el *data augmentation*, el dropout, L2, el optimizador y los *callbacks* del experimento 6.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 7: experimento 6 con Conv/Dense -> BatchNormalization -> LeakyReLU
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

modelo_exp07 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation,

    layers.Conv2D(64, (3, 3), use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(negative_slope=0.01),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(negative_slope=0.01),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(256, (3, 3), use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(negative_slope=0.01),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(128, use_bias=False, kernel_regularizer=regularizers.l2(L2_DENSA)),
    layers.BatchNormalization(),
    layers.LeakyReLU(negative_slope=0.01),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp07")
modelo_exp07.summary()

history_exp07 = funciones.entrenar_experimento(
    "exp07_bn_leakyrelu", modelo_exp07, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(50), batch_size=128, optimizer=Adam(learning_rate=1e-3),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=4, min_lr=1e-6, verbose=1),
    ],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 8: bloques Conv-BN dobles y GlobalAveragePooling**

**Objetivo e hipótesis:** en los modelos anteriores la mayor parte de los parámetros está en la conexión `Flatten` → `Dense`, que es la parte más propensa a memorizar. Se rediseña la red siguiendo un esquema tipo VGG: dos convoluciones 3×3 con `padding="same"` por bloque (mayor campo receptivo y más no linealidad con pocos parámetros), BN + LeakyReLU tras cada convolución y `GlobalAveragePooling2D` en lugar de `Flatten`, que reduce drásticamente el número de parámetros. Se espera mejor generalización con un modelo más pequeño.

**Cambios respecto al experimento anterior:** tres bloques dobles Conv-BN-LeakyReLU de 64, 128 y 256 filtros (`use_bias=False`), con `Dropout(0.3)` tras el segundo; `GlobalAveragePooling2D`; cabeza `Dense(256)`-BN-LeakyReLU-`Dropout(0.4)`-`Dense(256, L2=0.01)`-BN-LeakyReLU; *data augmentation* ampliado con `RandomContrast(0.1)` y `RandomTranslation(0.1, 0.1)`; Adam lr = 0.0005; `ReduceLROnPlateau(factor=0.5, patience=3)`; `EarlyStopping(patience=8)`; máximo 40 épocas (cada época es la más costosa de la sección). Al validar con un conjunto fijo (`validation_data`) ya no es necesario barajar manualmente los datos antes de entrenar (Keras baraja el entrenamiento en cada época).

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 8: bloques dobles Conv-BN-LeakyReLU + GlobalAveragePooling2D
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)


def bloque_conv_bn_leaky(filtros):
    """Dos Conv2D 3x3 (same, sin sesgo), cada una seguida de BN y LeakyReLU."""
    return [
        layers.Conv2D(filtros, (3, 3), padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(negative_slope=0.01),
        layers.Conv2D(filtros, (3, 3), padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(negative_slope=0.01),
    ]


data_augmentation_v2 = Sequential([   # augmentation ampliado
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation_v2")

modelo_exp08 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation_v2,

    *bloque_conv_bn_leaky(64),
    layers.MaxPooling2D((2, 2)),

    *bloque_conv_bn_leaky(128),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    *bloque_conv_bn_leaky(256),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),          # sustituye a Flatten
    layers.Dense(256, use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(negative_slope=0.01),
    layers.Dropout(0.4),
    layers.Dense(256, use_bias=False, kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.LeakyReLU(negative_slope=0.01),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp08")
modelo_exp08.summary()

history_exp08 = funciones.entrenar_experimento(
    "exp08_bloques_dobles_gap", modelo_exp08, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(40), batch_size=128, optimizer=Adam(learning_rate=5e-4),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 9: red compacta Conv-BN-ReLU con GlobalAveragePooling y sin capa densa oculta**

**Objetivo e hipótesis:** llevar al extremo la reducción de parámetros: cuatro bloques Conv-BN-ReLU con filtros crecientes (32 → 256), `GlobalAveragePooling2D` y la capa de salida directamente, sin capas densas ocultas. Con menos parámetros "memorizadores" y más profundidad convolucional se espera una generalización igual o mejor que la del experimento 8 con un coste menor. Se vuelve a ReLU estándar para comprobar si LeakyReLU es necesaria cuando se usa BN (la presencia de neuronas muertas se analiza en la sección 4.5).

**Cambios respecto al experimento anterior:** bloques dobles de 32, 64 y 128 filtros y un bloque simple de 256 (Conv `use_bias=False` → BN → `Activation("relu")`); `GlobalAveragePooling2D` → `Dropout(0.3)` → `Dense(15, softmax)`; *data augmentation* básico (volteo, rotación y zoom 0.1); Adam lr = 0.001; batch **32**; máximo 60 épocas; `EarlyStopping(patience=10)`; `ReduceLROnPlateau(factor=0.5, patience=4)`.

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión para el siguiente experimento:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 9: 4 bloques Conv-BN-ReLU + GlobalAveragePooling2D, sin capa densa oculta
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)


def conv_bn_relu(filtros):
    """Conv2D 3x3 (same, sin sesgo) + BatchNormalization + ReLU."""
    return [
        layers.Conv2D(filtros, (3, 3), padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
    ]


data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

modelo_exp09 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation,

    *conv_bn_relu(32), *conv_bn_relu(32),
    layers.MaxPooling2D((2, 2)),

    *conv_bn_relu(64), *conv_bn_relu(64),
    layers.MaxPooling2D((2, 2)),

    *conv_bn_relu(128), *conv_bn_relu(128),
    layers.MaxPooling2D((2, 2)),

    *conv_bn_relu(256),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="exp09")
modelo_exp09.summary()

history_exp09 = funciones.entrenar_experimento(
    "exp09_conv_bn_relu_gap", modelo_exp09, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(60), batch_size=32, optimizer=Adam(learning_rate=1e-3),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

**Experimento 10: modelo final regularizado**

**Objetivo e hipótesis:** sobre la red compacta del experimento 9, aumentar ligeramente la capacidad del último bloque y reforzar la regularización de forma combinada, para reducir el sobreajuste sin perder capacidad: `SpatialDropout2D` (anula mapas de características completos, más eficaz que el dropout estándar en capas convolucionales, donde los píxeles vecinos están correlacionados), un dropout final mayor, L2 en la capa de salida, *weight decay* desacoplado en Adam y un *data augmentation* algo más intenso.

**Cambios respecto al experimento anterior:** el bloque de 256 filtros pasa a ser doble; + `SpatialDropout2D(0.15)` tras el tercer bloque; Dropout final 0.3 → 0.45; L2 = 1e-4 en la capa de salida; `Adam(learning_rate=0.001, weight_decay=1e-4)`; *data augmentation* con rotación y zoom 0.15 y `RandomContrast(0.15)`; máximo 60 épocas, `EarlyStopping(patience=12)` y `ReduceLROnPlateau(factor=0.5, patience=5)` (paciencias mayores que en el experimento 9 porque la regularización adicional ralentiza la convergencia).

**Resultados observados:** [PENDIENTE: completar con las métricas de la ejecución]

**Interpretación y decisión:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Experimento 10: experimento 9 + SpatialDropout2D, más dropout, L2, weight decay y más augmentation
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

data_augmentation_v3 = Sequential([   # augmentation algo más intenso
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="data_augmentation_v3")

modelo_exp10 = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation_v3,

    *conv_bn_relu(32), *conv_bn_relu(32),
    layers.MaxPooling2D((2, 2)),

    *conv_bn_relu(64), *conv_bn_relu(64),
    layers.MaxPooling2D((2, 2)),

    *conv_bn_relu(128), *conv_bn_relu(128),
    layers.MaxPooling2D((2, 2)),
    layers.SpatialDropout2D(0.15),            # nuevo: dropout de mapas completos

    *conv_bn_relu(256), *conv_bn_relu(256),   # bloque de 256 ahora doble
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.45),                     # 0.3 -> 0.45
    layers.Dense(NUM_CLASSES, activation="softmax",
                 kernel_regularizer=regularizers.l2(1e-4)),
], name="exp10")
modelo_exp10.summary()

history_exp10 = funciones.entrenar_experimento(
    "exp10_final_regularizado", modelo_exp10, X_train_final, y_train_numerico, X_val, y_val,
    epochs=epocas(60), batch_size=32,
    optimizer=Adam(learning_rate=1e-3, weight_decay=1e-4),   # weight decay desacoplado
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ],
    resultados=resultados, historiales=historiales, models_dir=MODELS_DIR,
)

### Resumen de la experimentación *from scratch*

La tabla recoge, para cada experimento, la mejor `val_accuracy` y su época, la `val_loss` en esa época y la mínima alcanzada, la *accuracy* de entrenamiento en esa época y la diferencia con la de validación (`gap_acc`, indicador de sobreajuste; con *dropout* y *data augmentation* la *accuracy* de entrenamiento se mide con la regularización activa, por lo que el *gap* tiende a infraestimarse), el número de épocas entrenadas, los parámetros y el tiempo de entrenamiento. El gráfico compara la mejor `val_accuracy` de cada experimento con la de un clasificador aleatorio (1/15).

**Análisis de la progresión:** [PENDIENTE: completar con las métricas de la ejecución]

In [ ]:
# Tabla resumen de los experimentos (métricas de validación) y comparación gráfica
df_scratch = funciones.tabla_resultados(resultados, estrategia="From scratch")
display(df_scratch.drop(columns=["ruta_modelo"]).round(4))

funciones.graficar_resultados(
    df_scratch, metrica="best_val_accuracy", referencia=1 / NUM_CLASSES,
    titulo="Estrategia 1: mejor val_accuracy por experimento",
)

## 4.5 Selección del modelo *from scratch*

**Criterio:** se selecciona automáticamente el experimento con mayor `val_accuracy` en el conjunto de validación (en caso de empate, el de menor `val_loss`). El test interno no interviene en la selección: se reserva para la evaluación final de la sección 6. El modelo se recupera desde `modelos/<experimento>.keras` (pesos de su mejor época) y se comprueba que reproduce la `val_accuracy` registrada.

**Modelo seleccionado y justificación:** [PENDIENTE: identificar el modelo seleccionado y justificarlo a partir de los resultados de validación, la brecha entrenamiento-validación, el número de parámetros y el comportamiento de las curvas].

In [ ]:
# 4.5 Selección del mejor experimento por métricas de VALIDACIÓN y carga del modelo
df_scratch = funciones.tabla_resultados(resultados, estrategia="From scratch")
mejor_exp = df_scratch.sort_values(
    ["best_val_accuracy", "min_val_loss"], ascending=[False, True]
).iloc[0]
NOMBRE_SCRATCH_FINAL = mejor_exp["experimento"]
print(f"Experimento seleccionado: {NOMBRE_SCRATCH_FINAL} | "
      f"best val_accuracy={mejor_exp['best_val_accuracy']:.4f} (época {mejor_exp['best_epoch']}) | "
      f"parámetros={mejor_exp['parametros']:,}")

modelo_scratch_final = tf.keras.models.load_model(mejor_exp["ruta_modelo"])

# Comprobación: el modelo recuperado reproduce la val_accuracy registrada
val_loss_final, val_acc_final = modelo_scratch_final.evaluate(X_val, y_val, batch_size=128, verbose=0)
print(f"Modelo recuperado -> val_accuracy={val_acc_final:.4f} | val_loss={val_loss_final:.4f}")

h = historiales[NOMBRE_SCRATCH_FINAL]
funciones.impresion(range(1, len(h["accuracy"]) + 1), h,
                    titulo=f"Modelo from scratch seleccionado: {NOMBRE_SCRATCH_FINAL}")

### Diagnóstico de neuronas muertas del modelo seleccionado

Con ReLU, una neurona cuya entrada es negativa para todos los ejemplos produce siempre 0 y su gradiente es nulo, por lo que deja de aprender ("neurona muerta"). Se pasan las primeras 256 imágenes de validación por el modelo seleccionado (en modo inferencia, sin *dropout* ni *augmentation*) y se cuenta, en cada capa con activación ReLU, cuántos filtros/neuronas valen 0 para todas las imágenes y posiciones. En capas LeakyReLU, que nunca anulan el gradiente, se informa de las unidades que nunca producen salida positiva ("inactivas").

**Resultado y comentario:** [PENDIENTE: comentar el porcentaje de neuronas muertas o inactivas por capa y si condiciona la elección de la activación].

In [ ]:
# Diagnóstico de neuronas muertas/inactivas con las primeras 256 imágenes de validación
diagnostico_neuronas = funciones.diagnosticar_neuronas_muertas(modelo_scratch_final, X_val[:256])

# 5. Estrategia 2 — Modelos preentrenados

En esta sección se comparan **dos arquitecturas CNN preentrenadas sobre ImageNet**, **EfficientNetB0** y **ResNet50V2** (ambas de `tf.keras.applications`). Para cada una se documentan dos fases consecutivas:

- **Fase A — *Transfer learning*:** la base convolucional permanece congelada y solo se entrena un clasificador nuevo; la red actúa como extractor de características genéricas aprendidas en ImageNet.
- **Fase B — *Fine-tuning*:** partiendo de los pesos de la fase A, se descongela selectivamente la parte superior de la base y se reentrena con una tasa de aprendizaje muy baja para adaptar las características de alto nivel al problema HAR.

Se utilizan **exactamente las mismas particiones** (`idx_tr`, `idx_val`, `idx_test`) que en la estrategia *from scratch*, de modo que la comparación de la sección 6 sea justa. La partición de test interno **no se usa** en esta sección: todas las decisiones (*early stopping*, profundidad de descongelado, elección de arquitectura) se toman con la partición de validación.

## 5.1 Decisiones comunes

- **Resolución de entrada: 224×224 px.** Es la resolución nativa con la que se preentrenaron ambas arquitecturas en ImageNet; usarla evita el desajuste de escala entre los filtros aprendidos y los objetos de la imagen. Las imágenes originales, de tamaño variable (estadísticas en la sección 2.5), se reescalan a esta resolución, que conserva más detalle que los 128×128 px usados en la CNN *from scratch* (elegidos allí por coste computacional).
- **Preprocesamiento específico de cada arquitectura:** el `tf.data` entrega imágenes en rango **0–255** (`float32`) y cada modelo incorpora su propio preprocesado como primera capa, de forma que **un único pipeline alimenta a ambas redes**:
  - *EfficientNetB0*: la red de Keras ya incluye internamente las capas `Rescaling` y `Normalization` con las estadísticas de ImageNet; `efficientnet.preprocess_input` es la identidad, por lo que espera directamente valores 0–255.
  - *ResNet50V2*: `resnet_v2.preprocess_input` escala los píxeles a [-1, 1] (`x/127.5 − 1`). Se implementa con una capa `Rescaling(1/127.5, offset=-1)` equivalente (se comprueba numéricamente más abajo), lo que mantiene el modelo serializable en formato `.keras` sin capas `Lambda`.
- **Pesos iniciales:** ImageNet (`weights="imagenet"`), sin la cabeza original de 1000 clases (`include_top=False`).
- **Estrategia para congelar/descongelar capas:** en la fase A toda la base está congelada (`base.trainable = False`) y se invoca con `training=False`. En la fase B se descongela desde un bloque determinado hasta el final (descongelado selectivo de los bloques superiores, que codifican rasgos más específicos de la tarea original), manteniendo **congeladas todas las capas `BatchNormalization`**. Para EfficientNetB0 se comparan dos profundidades de descongelado.
- **Clasificador final:** `GlobalAveragePooling2D → Dropout(0.3) → Dense(15, softmax)`. El *pooling* global reduce el mapa de características a un vector (1280 componentes en EfficientNetB0, 2048 en ResNet50V2) sin añadir parámetros; se descarta una capa densa oculta adicional porque estas características ya son de alto nivel y, con ≈8800 imágenes de entrenamiento, una capa extra solo añadiría parámetros y riesgo de sobreajuste (lo observado en la estrategia *from scratch*).
- **Regularización y *data augmentation*:** `Dropout(0.3)` antes de la capa de salida; *data augmentation* ligera dentro del modelo (`RandomFlip` horizontal, `RandomRotation(±0.05·2π ≈ ±18°)`, `RandomZoom(±10 %)`), activa solo durante el entrenamiento. No se usa volteo vertical porque las acciones humanas tienen una orientación canónica (personas de pie/sentadas). *Early stopping* sobre `val_loss` con restauración de los mejores pesos y `ReduceLROnPlateau` completan la regularización.

Las dos arquitecturas comparten, por tanto, el pipeline de datos, la cabeza, la regularización y el protocolo de entrenamiento; las únicas diferencias son la base convolucional y su capa de preprocesado. Así, las diferencias de rendimiento se pueden atribuir a la arquitectura y no a otros factores.

In [ ]:
# 5.1 Configuración común y pipeline tf.data para los modelos preentrenados
import time
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.applications import efficientnet, resnet_v2

AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE_PRE = (224, 224)          # resolución nativa de ImageNet
BATCH_SIZE_PRE = 32                # 224x224 + fine-tuning de ResNet50V2: lote moderado por memoria
EPOCHS_TL = 2 if QUICK_RUN else 15     # máximo de épocas fase A (EarlyStopping decide)
EPOCHS_FT = 2 if QUICK_RUN else 12     # máximo de épocas fase B
LR_TL, LR_FT = 1e-3, 1e-5
os.makedirs(MODELS_DIR, exist_ok=True)

# Etiquetas codificadas con el mismo LabelEncoder de la sección 4
y_all = encoder.transform(df_train["label"].values).astype("int32")
rutas_all = np.array([os.path.join(TRAIN_DIR, f) for f in df_train["filename"].values])
# Mismas filas y etiquetas que los arrays de la sección 3: idx_tr/idx_val/idx_test son intercambiables
assert len(rutas_all) == len(y_codificadas) and np.array_equal(y_all, y_codificadas)

def _cargar_imagen(ruta, etiqueta):
    """Lee el JPG, lo decodifica a RGB y lo redimensiona a 224x224 (uint8 para cachear)."""
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE_PRE, method="bilinear", antialias=True)
    img = tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)
    return img, etiqueta

def crear_dataset(indices, entrenamiento=False):
    """Dataset tf.data a partir de rutas. Cachea en memoria las imágenes ya
    redimensionadas en uint8 (≈1.9 GB para las 12600 imágenes) y entrega
    lotes float32 en rango 0-255. El orden se conserva si entrenamiento=False."""
    ds = tf.data.Dataset.from_tensor_slices((rutas_all[indices], y_all[indices]))
    ds = ds.map(_cargar_imagen, num_parallel_calls=AUTOTUNE).cache()
    if entrenamiento:
        ds = ds.shuffle(len(indices), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE_PRE)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

ds_train_pre = crear_dataset(idx_tr, entrenamiento=True)
ds_val_pre = crear_dataset(idx_val)
ds_test_pre = crear_dataset(idx_test)   # solo se usará en la sección 6

print(f"Train: {len(idx_tr)}  Val: {len(idx_val)}  Test interno: {len(idx_test)}  "
      f"(lotes de {BATCH_SIZE_PRE}, entrada {IMG_SIZE_PRE})")
# Lote de ejemplo desde un pipeline SIN caché (leer solo un lote de un dataset cacheado
# descartaría la caché parcial y generaría avisos de TensorFlow)
ds_muestra = (tf.data.Dataset.from_tensor_slices((rutas_all[idx_val[:BATCH_SIZE_PRE]], y_all[idx_val[:BATCH_SIZE_PRE]]))
              .map(_cargar_imagen).batch(BATCH_SIZE_PRE)
              .map(lambda x, y: (tf.cast(x, tf.float32), y)))
xb, yb = next(iter(ds_muestra))
print("Lote de ejemplo:", xb.shape, xb.dtype, f"rango [{float(tf.reduce_min(xb)):.0f}, {float(tf.reduce_max(xb)):.0f}]")

# Comprobación: Rescaling(1/127.5, offset=-1) == resnet_v2.preprocess_input
_ref = resnet_v2.preprocess_input(xb.numpy().copy())
_cap = layers.Rescaling(1 / 127.5, offset=-1)(xb).numpy()
print("Rescaling equivalente a resnet_v2.preprocess_input:", np.allclose(_ref, _cap, atol=1e-5))
print("efficientnet.preprocess_input es la identidad:",
      np.allclose(efficientnet.preprocess_input(xb.numpy().copy()), xb.numpy()))

A continuación se definen las funciones reutilizables comunes a ambas arquitecturas, para evitar duplicar código: construcción del modelo, descongelado selectivo, entrenamiento con callbacks, representación de curvas por fases y registro de resultados. Cada fase se ejecuta después en su propia celda para que quede documentada de forma independiente.

In [ ]:
# Funciones reutilizables para los modelos preentrenados

ARQUITECTURAS = {
    "EfficientNetB0": {"constructor": EfficientNetB0, "preprocesado": None},
    "ResNet50V2": {"constructor": ResNet50V2,
                   "preprocesado": lambda: layers.Rescaling(1 / 127.5, offset=-1, name="preprocess_resnet_v2")},
}

def crear_aumentacion():
    """Data augmentation ligera; Keras la desactiva automáticamente en inferencia."""
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.05, seed=SEED),
        layers.RandomZoom(0.1, seed=SEED),
    ], name="data_augmentation")

def construir_modelo(arquitectura, dropout=0.3):
    """Base ImageNet congelada + cabeza GAP -> Dropout -> Dense(15)."""
    tf.keras.utils.set_random_seed(SEED)
    cfg = ARQUITECTURAS[arquitectura]
    base = cfg["constructor"](include_top=False, weights="imagenet", input_shape=IMG_SIZE_PRE + (3,))
    base.trainable = False
    entradas = layers.Input(shape=IMG_SIZE_PRE + (3,), name="imagen_0_255")
    x = crear_aumentacion()(entradas)
    if cfg["preprocesado"] is not None:
        x = cfg["preprocesado"]()(x)
    # training=False: las BatchNormalization de la base trabajan siempre en modo
    # inferencia (estadísticas de ImageNet), también durante el fine-tuning.
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout, name="dropout_cabeza")(x)
    salidas = layers.Dense(NUM_CLASSES, activation="softmax", name="clasificador")(x)
    return Model(entradas, salidas, name=f"{arquitectura}_HAR")

def obtener_base(modelo):
    return next(l for l in modelo.layers if isinstance(l, tf.keras.Model) and l.name != "data_augmentation")

def contar_parametros(modelo):
    entrenables = int(sum(np.prod(w.shape) for w in modelo.trainable_weights))
    return entrenables, int(modelo.count_params())

def descongelar_desde(modelo, prefijo_capa):
    """Descongela la base desde la primera capa cuyo nombre empieza por
    `prefijo_capa` hasta el final, manteniendo congeladas las BatchNormalization."""
    base = obtener_base(modelo)
    base.trainable = True
    inicio = next(i for i, l in enumerate(base.layers) if l.name.startswith(prefijo_capa))
    for i, capa in enumerate(base.layers):
        capa.trainable = (i >= inicio) and not isinstance(capa, layers.BatchNormalization)
    n_desc = sum(1 for capa in base.layers if capa.trainable and capa.weights)
    print(f"Descongeladas {n_desc} capas con pesos de {len(base.layers)} capas de la base "
          f"(desde '{base.layers[inicio].name}', índice {inicio}); BatchNormalization congeladas.")
    return inicio

def entrenar_fase(modelo, lr, epochs, initial_epoch=0, paciencia=4):
    """Compila y entrena; devuelve (history_dict, tiempo_s)."""
    modelo.compile(optimizer=Adam(learning_rate=lr), loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"], jit_compile=False)
    cbs = [EarlyStopping(monitor="val_loss", patience=paciencia, restore_best_weights=True, verbose=1),
           ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1)]
    t0 = time.perf_counter()
    h = modelo.fit(ds_train_pre, validation_data=ds_val_pre, epochs=initial_epoch + epochs,
                   initial_epoch=initial_epoch, callbacks=cbs, verbose=2)
    return h.history, time.perf_counter() - t0

def concatenar_historiales(h_tl, h_ft):
    return {k: list(h_tl[k]) + list(h_ft[k]) for k in ["accuracy", "val_accuracy", "loss", "val_loss"]}

def graficar_fases(h_tl, h_fts, titulo):
    """Curvas TL + una o varias continuaciones de FT, con línea vertical en el cambio de fase."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    n_tl = len(h_tl["loss"])
    ep_tl = np.arange(1, n_tl + 1)
    colores = ["tab:green", "tab:red", "tab:purple"]
    for ax, met in zip(axes, ["accuracy", "loss"]):
        ax.plot(ep_tl, h_tl[met], "o-", color="tab:blue", label="TL entrenamiento")
        ax.plot(ep_tl, h_tl["val_" + met], "s--", color="tab:blue", alpha=0.7, label="TL validación")
        for c, (nombre, h_ft) in zip(colores, h_fts.items()):
            # Se antepone el último punto de TL para que cada curva FT continúe la de TL
            ep_ft = np.arange(n_tl, n_tl + len(h_ft["loss"]) + 1)
            ax.plot(ep_ft, [h_tl[met][-1]] + list(h_ft[met]), "o-", color=c, label=f"{nombre} entrenamiento")
            ax.plot(ep_ft, [h_tl["val_" + met][-1]] + list(h_ft["val_" + met]), "s--", color=c, alpha=0.7,
                    label=f"{nombre} validación")
        ax.axvline(n_tl + 0.5, color="gray", linestyle=":", label="inicio fine-tuning")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_xlabel("Época"); ax.set_ylabel(met)
        ax.set_title(f"{titulo} — {'Precisión' if met == 'accuracy' else 'Pérdida'}", weight="bold")
        ax.grid(True, linestyle="--", alpha=0.6); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

# Almacenes locales de esta sección
modelos_pre = {}       # nombre experimento -> modelo Keras entrenado
info_pre = {}          # nombre experimento -> {arquitectura, historial_fase, historial_total, tiempo_total_s, ...}

def registrar_pre(experimento, arquitectura, historial_fase, modelo, tiempo_fase_s,
                  epoca_offset=0, tiempo_previo_s=0.0):
    """Añade la fase entrenada al registro común `resultados` (funciones.registrar_resultado),
    guarda el modelo y su historial e imprime un resumen de una línea.
    - Las métricas se calculan sobre la fase entrenada (los pesos retenidos pertenecen a ella).
    - best_epoch se expresa en la numeración global (TL + FT) mediante `epoca_offset`.
    - tiempo_s / s_por_epoca corresponden solo a esta fase; tiempo_acumulado_s suma la fase A previa."""
    h = historial_fase
    ruta = os.path.join(MODELS_DIR, f"{experimento}.keras")
    mejor = int(np.argmax(h["val_accuracy"]))
    fila = funciones.registrar_resultado(
        resultados, "Preentrenada", experimento, h, modelo, tiempo_fase_s, ruta_modelo=ruta,
        best_epoch=epoca_offset + mejor + 1, arquitectura=arquitectura,
        tiempo_acumulado_s=round(float(tiempo_previo_s + tiempo_fase_s), 1))
    historiales[experimento] = dict(h)
    modelos_pre[experimento] = modelo
    modelo.save(ruta)
    print(f"[{experimento}] best val_accuracy={fila['best_val_accuracy']:.4f} "
          f"(época {fila['best_epoch']}) | min val_loss={fila['min_val_loss']:.4f} | "
          f"épocas en esta fase={fila['epochs_entrenadas']} | parámetros={fila['parametros']:,} | "
          f"tiempo fase={fila['tiempo_s']:.0f} s ({fila['s_por_epoca']:.1f} s/época)")
    return fila

## 5.2 Arquitectura preentrenada 1 — EfficientNetB0

### Selección y descripción

**EfficientNetB0** (Tan & Le, 2019) es el modelo base de la familia EfficientNet, obtenido mediante búsqueda de arquitectura neuronal y escalado compuesto equilibrado de profundidad, anchura y resolución. Se construye con bloques *MBConv* (convoluciones *inverted residual* separables en profundidad) con módulos *squeeze-and-excitation*. Según la documentación de Keras Applications tiene ≈5.3 M de parámetros (≈4.0 M sin la cabeza) y un 77.1 % de *top-1* en ImageNet, lo que la convierte en una de las redes con mejor relación precisión/coste. Se elige como representante de las arquitecturas **ligeras y eficientes**: pocos parámetros que ajustar con un dataset de tamaño medio y entrenamiento rápido a 224×224 px.

### Fase A — Transfer learning

Toda la base convolucional permanece congelada (`trainable=False`) y solo se entrenan los pesos de la capa `Dense` final (≈19 k parámetros). La base actúa como extractor fijo de características de ImageNet, que son muy informativas para fotografías de personas y objetos cotidianos. Se usa Adam con `lr=1e-3` (valor estándar, adecuado porque solo se entrena una cabeza inicializada aleatoriamente), un máximo de 15 épocas, *early stopping* sobre `val_loss` (paciencia 4, restaurando los mejores pesos) y `ReduceLROnPlateau` (factor 0.2, paciencia 2).

In [ ]:
# 5.2 EfficientNetB0 — construcción del modelo (base congelada)
modelo_eff_tl = construir_modelo("EfficientNetB0")
modelo_eff_tl.summary()
print("Parámetros entrenables / totales:", contar_parametros(modelo_eff_tl))

In [ ]:
# 5.2 EfficientNetB0 — Fase A: transfer learning
hist_eff_tl, t_eff_tl = entrenar_fase(modelo_eff_tl, LR_TL, EPOCHS_TL)
registrar_pre("EfficientNetB0-TL", "EfficientNetB0", hist_eff_tl, modelo_eff_tl, t_eff_tl)
info_pre["EfficientNetB0-TL"] = {"arquitectura": "EfficientNetB0", "historial_total": hist_eff_tl,
                                 "tiempo_total_s": t_eff_tl}
# Pesos de la fase A: punto de partida común para todas las variantes de fine-tuning
PESOS_EFF_TL = os.path.join(MODELS_DIR, "EfficientNetB0-TL.weights.h5")
modelo_eff_tl.save_weights(PESOS_EFF_TL)

### Monitorización de transfer learning

[PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* valor y época de la mejor `val_accuracy`/`val_loss`; distancia entre curvas de entrenamiento y validación (con *dropout* y *augmentation* activos, la precisión de entrenamiento puede quedar por debajo de la de validación); si la curva de validación se ha estabilizado (lo que motiva la fase B) y si actuaron `ReduceLROnPlateau`/`EarlyStopping`.

In [ ]:
# 5.2 EfficientNetB0 — curvas de la fase A
funciones.impresion(range(1, len(hist_eff_tl["accuracy"]) + 1), hist_eff_tl,
                    titulo="EfficientNetB0 — fase A (transfer learning)")

### Fase B — Fine-tuning

Partiendo de los pesos de la fase A, se descongelan los bloques superiores de la base y se reentrena con **Adam `lr=1e-5`** (100 veces menor que en la fase A), para ajustar ligeramente los filtros de alto nivel sin destruir las características preentrenadas (*catastrophic forgetting*). Es imprescindible haber entrenado antes la cabeza: si se descongela la base con una cabeza aleatoria, los gradientes iniciales, grandes, degradarían los pesos de ImageNet.

- **Capas `BatchNormalization` congeladas:** sus estadísticas (media y varianza móviles) se estimaron sobre millones de imágenes de ImageNet. Con lotes de 32 imágenes de un dataset pequeño, actualizarlas introduce ruido y desplaza la distribución de activaciones que esperan las capas posteriores, lo que suele provocar un empeoramiento brusco de la validación. Por ello la base se llama con `training=False` y, además, sus capas BN se mantienen con `trainable=False`.
- **Descongelado selectivo, comparando dos profundidades** (se continúa la numeración de épocas con `initial_epoch`):
  - **FT-parcial:** desde `block6a` hasta el final (bloques 6–7 y `top_conv`, la parte más específica de la tarea original).
  - **FT-amplio:** desde `block4a` hasta el final (aproximadamente la mitad superior de la red), con más capacidad de adaptación pero más riesgo de sobreajuste.

Ambas variantes parten **de los mismos pesos de la fase A**, de modo que la única diferencia es la profundidad descongelada.

In [ ]:
# 5.2 EfficientNetB0 — Fase B: fine-tuning con dos profundidades de descongelado
CONFIG_FT_EFF = {"EfficientNetB0-FT-parcial": "block6a", "EfficientNetB0-FT-amplio": "block4a"}
n_ep_eff_tl = len(hist_eff_tl["loss"])
hist_eff_ft = {}

for nombre, prefijo in CONFIG_FT_EFF.items():
    print(f"\n===== {nombre} (descongelado desde {prefijo}) =====")
    modelo_ft = construir_modelo("EfficientNetB0")
    modelo_ft.load_weights(PESOS_EFF_TL)
    descongelar_desde(modelo_ft, prefijo)
    print("Parámetros entrenables / totales:", contar_parametros(modelo_ft))
    h_ft, t_ft = entrenar_fase(modelo_ft, LR_FT, EPOCHS_FT, initial_epoch=n_ep_eff_tl)
    hist_eff_ft[nombre] = h_ft
    registrar_pre(nombre, "EfficientNetB0", h_ft, modelo_ft, t_ft,
                  epoca_offset=n_ep_eff_tl, tiempo_previo_s=t_eff_tl)
    # El coste total de una variante FT incluye el de su fase A previa (tiempo acumulado)
    info_pre[nombre] = {"arquitectura": "EfficientNetB0", "historial_total": concatenar_historiales(hist_eff_tl, h_ft),
                        "tiempo_total_s": t_eff_tl + t_ft}

In [ ]:
# 5.2 EfficientNetB0 — curvas combinadas de las fases A y B
graficar_fases(hist_eff_tl, hist_eff_ft, "EfficientNetB0")

### Análisis de la arquitectura

[PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* ganancia de `val_accuracy`/`val_loss` de cada variante de *fine-tuning* respecto a la fase A; si el descongelado amplio aporta mejora adicional o, por el contrario, aumenta la separación entre entrenamiento y validación (sobreajuste); coste temporal añadido de cada variante; configuración que se retiene para EfficientNetB0 y por qué.

## 5.3 Arquitectura preentrenada 2 — ResNet50V2

### Selección y descripción

**ResNet50V2** (He et al., 2016b) es la versión "v2" de ResNet-50, que reorganiza cada bloque residual en modo *pre-activación* (BatchNormalization → ReLU → convolución) para que la conexión identidad propague la señal y el gradiente sin transformaciones, facilitando la optimización de redes profundas. Según Keras Applications tiene ≈25.6 M de parámetros (≈23.6 M sin la cabeza) y un 76.0 % de *top-1* en ImageNet. Se elige como contrapunto a EfficientNetB0: una arquitectura **clásica, más ancha y ≈6 veces más pesada**, que produce un vector de características de 2048 componentes. Permite estudiar si una mayor capacidad compensa su mayor coste computacional en este problema.

### Fase A — Transfer learning

Mismo protocolo que en EfficientNetB0: base completamente congelada, cabeza `GAP → Dropout(0.3) → Dense(15)` (≈31 k parámetros entrenables), Adam `lr=1e-3`, máximo 15 épocas, *early stopping* sobre `val_loss` y `ReduceLROnPlateau`. La única diferencia es la capa de preprocesado a [-1, 1] previa a la base.

In [ ]:
# 5.3 ResNet50V2 — construcción del modelo (base congelada)
modelo_res_tl = construir_modelo("ResNet50V2")
modelo_res_tl.summary()
print("Parámetros entrenables / totales:", contar_parametros(modelo_res_tl))

In [ ]:
# 5.3 ResNet50V2 — Fase A: transfer learning
hist_res_tl, t_res_tl = entrenar_fase(modelo_res_tl, LR_TL, EPOCHS_TL)
registrar_pre("ResNet50V2-TL", "ResNet50V2", hist_res_tl, modelo_res_tl, t_res_tl)
info_pre["ResNet50V2-TL"] = {"arquitectura": "ResNet50V2", "historial_total": hist_res_tl,
                             "tiempo_total_s": t_res_tl}
PESOS_RES_TL = os.path.join(MODELS_DIR, "ResNet50V2-TL.weights.h5")
modelo_res_tl.save_weights(PESOS_RES_TL)

### Monitorización de transfer learning

[PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* mismos puntos que en EfficientNetB0 (mejor época, separación entre curvas, estabilización) y comparación directa con la fase A de EfficientNetB0 con el mismo protocolo.

In [ ]:
# 5.3 ResNet50V2 — curvas de la fase A
funciones.impresion(range(1, len(hist_res_tl["accuracy"]) + 1), hist_res_tl,
                    titulo="ResNet50V2 — fase A (transfer learning)")

### Fase B — Fine-tuning

Se descongela la **etapa 5 completa** (desde `conv5_block1` hasta el final de la base), manteniendo congeladas las BatchNormalization por los motivos expuestos en 5.2, y se reentrena con Adam `lr=1e-5` continuando la numeración de épocas. En ResNet50V2 esta etapa ya concentra ≈15 M de parámetros (casi dos tercios de la base), por lo que descongelar más profundidad multiplicaría el coste y el riesgo de sobreajuste con ≈8800 imágenes de entrenamiento; se adopta, por tanto, una única profundidad, apoyándose en la comparación de profundidades realizada con EfficientNetB0.

In [ ]:
# 5.3 ResNet50V2 — Fase B: fine-tuning de la etapa conv5
CONFIG_FT_RES = {"ResNet50V2-FT-conv5": "conv5_block1"}
n_ep_res_tl = len(hist_res_tl["loss"])
hist_res_ft = {}

for nombre, prefijo in CONFIG_FT_RES.items():
    print(f"\n===== {nombre} (descongelado desde {prefijo}) =====")
    modelo_ft = construir_modelo("ResNet50V2")
    modelo_ft.load_weights(PESOS_RES_TL)
    descongelar_desde(modelo_ft, prefijo)
    print("Parámetros entrenables / totales:", contar_parametros(modelo_ft))
    h_ft, t_ft = entrenar_fase(modelo_ft, LR_FT, EPOCHS_FT, initial_epoch=n_ep_res_tl)
    hist_res_ft[nombre] = h_ft
    registrar_pre(nombre, "ResNet50V2", h_ft, modelo_ft, t_ft,
                  epoca_offset=n_ep_res_tl, tiempo_previo_s=t_res_tl)
    info_pre[nombre] = {"arquitectura": "ResNet50V2", "historial_total": concatenar_historiales(hist_res_tl, h_ft),
                        "tiempo_total_s": t_res_tl + t_ft}

In [ ]:
# 5.3 ResNet50V2 — curvas combinadas de las fases A y B
graficar_fases(hist_res_tl, hist_res_ft, "ResNet50V2")

### Análisis de la arquitectura

[PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* mejora (o no) del *fine-tuning* frente al *transfer learning*; evidencias de sobreajuste al descongelar ≈15 M de parámetros; coste temporal por época frente a EfficientNetB0; configuración que se retiene para ResNet50V2.

## 5.4 Comparación de arquitecturas preentrenadas

Ambas arquitecturas se comparan con las mismas particiones, el mismo pipeline y las mismas métricas. Como `EarlyStopping` restaura los pesos con menor `val_loss` (que no tiene por qué coincidir con la época de mayor `val_accuracy` del historial), cada modelo retenido se **reevalúa sobre la partición de validación** y la selección se hace con esa precisión, que corresponde exactamente a los pesos que se usarán después. El test interno sigue sin utilizarse.

In [ ]:
# 5.4 Tabla comparativa de las variantes preentrenadas (solo validación)
def entropia_cruzada(y, prob):
    """Sparse categorical cross-entropy media calculada a partir de probabilidades."""
    return float(-np.mean(np.log(np.clip(prob[np.arange(len(y)), y], 1e-7, 1.0))))

y_val_pre = y_all[idx_val]
filas_cmp = []
for nombre, modelo in modelos_pre.items():
    prob_val = modelo.predict(ds_val_pre, verbose=0)
    val_acc = float(np.mean(prob_val.argmax(axis=1) == y_val_pre))
    val_loss = entropia_cruzada(y_val_pre, prob_val)
    entrenables, totales = contar_parametros(modelo)
    h = info_pre[nombre]["historial_total"]
    filas_cmp.append({"experimento": nombre, "arquitectura": info_pre[nombre]["arquitectura"],
                      "val_accuracy (pesos retenidos)": val_acc, "val_loss (pesos retenidos)": val_loss,
                      "épocas totales": len(h["loss"]), "parámetros totales": totales,
                      "parámetros entrenables (última fase)": entrenables,
                      "tiempo acumulado (min)": info_pre[nombre]["tiempo_total_s"] / 60})
    info_pre[nombre]["val_accuracy"] = val_acc
    info_pre[nombre]["val_loss"] = val_loss
# Orden por val_accuracy de los pesos retenidos; en caso de empate, menor val_loss (mismo criterio que en 4.5)
df_cmp_pre = pd.DataFrame(filas_cmp).sort_values(["val_accuracy (pesos retenidos)", "val_loss (pesos retenidos)"],
                                                 ascending=[False, True])
display(df_cmp_pre.round(4))

# Registro común (mismo formato que la sección 4) filtrado a la estrategia preentrenada.
# best_val_accuracy es el máximo del historial de cada fase; la tabla anterior refleja los pesos retenidos.
display(funciones.tabla_resultados(resultados, estrategia="Preentrenada")
        .drop(columns=["ruta_modelo"]).round(4))

# Gráfico de barras: precisión de validación y coste temporal
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
colores_arq = df_cmp_pre["arquitectura"].map({"EfficientNetB0": "tab:green", "ResNet50V2": "tab:orange"})
ax1.bar(df_cmp_pre["experimento"], df_cmp_pre["val_accuracy (pesos retenidos)"], color=colores_arq)
ax1.set_title("Precisión de validación por variante", weight="bold"); ax1.set_ylabel("val_accuracy")
ax1.set_ylim(0, 1); ax1.tick_params(axis="x", rotation=30)
for i, v in enumerate(df_cmp_pre["val_accuracy (pesos retenidos)"]):
    ax1.text(i, v + 0.01, f"{v:.3f}", ha="center")
ax2.bar(df_cmp_pre["experimento"], df_cmp_pre["tiempo acumulado (min)"], color=colores_arq)
ax2.set_title("Tiempo de entrenamiento acumulado", weight="bold"); ax2.set_ylabel("minutos")
ax2.tick_params(axis="x", rotation=30)
for ax in (ax1, ax2):
    ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout(); plt.show()

In [ ]:
# 5.4 Selección (por validación) del mejor modelo de cada arquitectura y del mejor global
mejor_por_arq = df_cmp_pre.groupby("arquitectura", sort=False).head(1).set_index("arquitectura")["experimento"].to_dict()
NOMBRE_PRE_FINAL = df_cmp_pre.iloc[0]["experimento"]
modelo_pre_final = modelos_pre[NOMBRE_PRE_FINAL]
print("Mejor variante por arquitectura (validación):", mejor_por_arq)
print(f"Modelo preentrenado seleccionado: {NOMBRE_PRE_FINAL} "
      f"(val_accuracy = {info_pre[NOMBRE_PRE_FINAL]['val_accuracy']:.4f})")

**Análisis comparativo:** [PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* arquitectura y variante con mejor precisión de validación; diferencia frente a la otra arquitectura en relación con su número de parámetros y su tiempo de entrenamiento (rendimiento por unidad de coste); comportamiento de generalización (separación entre curvas); justificación de la elección de `modelo_pre_final`.

# 6. Evaluación y comparación global de estrategias

## 6.1 Evaluación final sobre test

En esta sección se utiliza **por primera vez** la partición de test interno (15 %, `idx_test`), que no ha intervenido en el entrenamiento ni en ninguna decisión de selección. Se evalúan únicamente los modelos ya seleccionados con validación:

- el mejor modelo *from scratch* (`modelo_scratch_final`, entrada 128×128 normalizada a [0, 1]);
- el mejor modelo de cada arquitectura preentrenada (entrada 224×224, mismo orden de imágenes que `X_test_interno`, sin barajar).

Como referencia mínima se incluyen dos clasificadores triviales: predicción aleatoria uniforme (precisión esperada 1/15 ≈ 6.7 %) y predicción de la clase mayoritaria del entrenamiento (en un dataset balanceado, también ≈ 6.7 %).

In [ ]:
# 6.1 Evaluación final sobre el test interno
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

y_test_ref = y_all[idx_test]
# Comprobaciones de integridad: el test interno es disjunto de entrenamiento y validación, y el
# pipeline 224x224 (sin barajar) entrega exactamente las mismas imágenes y en el mismo orden
# que X_test_interno (128x128) de la sección 3
assert not set(idx_test) & (set(idx_tr) | set(idx_val))
assert np.array_equal(y_test_ref, np.asarray(y_test_interno)), \
    "El orden de idx_test no coincide con y_test_interno: revisar la partición de la sección 3"
assert np.array_equal(np.concatenate([y.numpy() for _, y in ds_test_pre]), y_test_ref), \
    "ds_test_pre no conserva el orden de idx_test"
print(f"Test interno: {len(y_test_ref)} imágenes, no usadas en entrenamiento ni selección.")

def resumen_test(nombre, estrategia, modelo, datos_x):
    """Predice sobre test (array o tf.data sin barajar) y calcula accuracy, F1-macro y
    pérdida (entropía cruzada) a partir de las probabilidades predichas."""
    prob = modelo.predict(datos_x, batch_size=128, verbose=0) if isinstance(datos_x, np.ndarray) \
        else modelo.predict(datos_x, verbose=0)
    pred = prob.argmax(axis=1)
    return {"modelo": nombre, "estrategia": estrategia, "test_accuracy": accuracy_score(y_test_ref, pred),
            "test_macro_f1": f1_score(y_test_ref, pred, average="macro"),
            "test_loss": entropia_cruzada(y_test_ref, prob),
            "parametros": int(modelo.count_params())}, pred

eval_test, preds_test = [], {}
r, preds_test[NOMBRE_SCRATCH_FINAL] = resumen_test(NOMBRE_SCRATCH_FINAL, "From scratch", modelo_scratch_final,
                                                   np.asarray(X_test_interno))
eval_test.append(r)
for arq, nombre in mejor_por_arq.items():
    r, preds_test[nombre] = resumen_test(nombre, "Preentrenada", modelos_pre[nombre], ds_test_pre)
    eval_test.append(r)

# Clasificadores triviales de referencia
rng = np.random.default_rng(SEED)
pred_aleatoria = rng.integers(0, NUM_CLASSES, size=len(y_test_ref))
clase_mayoritaria = np.bincount(y_all[idx_tr]).argmax()
pred_mayoritaria = np.full_like(y_test_ref, clase_mayoritaria)
for nombre, pred in [("Aleatorio uniforme", pred_aleatoria), ("Clase mayoritaria", pred_mayoritaria)]:
    eval_test.append({"modelo": nombre, "estrategia": "Trivial", "test_accuracy": accuracy_score(y_test_ref, pred),
                      "test_macro_f1": f1_score(y_test_ref, pred, average="macro"), "test_loss": np.nan,
                      "parametros": 0})

df_eval_test = pd.DataFrame(eval_test)
display(df_eval_test.round(4))
print(f"Precisión esperada de un clasificador aleatorio: 1/{NUM_CLASSES} = {1 / NUM_CLASSES:.4f}")

[PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* precisión y F1-macro en test de cada modelo frente a los clasificadores triviales (cuántas veces superan el 6.7 %); coherencia entre la precisión de validación y la de test (si la diferencia es pequeña, la selección con validación no ha sobreajustado a esa partición).

## 6.2 Métricas y análisis por clase

Al estar el dataset balanceado, la *accuracy* es una métrica representativa, pero se complementa con *precision*, *recall* y F1 por clase y con la matriz de confusión normalizada por filas (cada fila suma 1 y su diagonal es el *recall* de la clase), para identificar qué acciones se confunden entre sí. Se analizan el mejor modelo *from scratch* y el mejor preentrenado.

In [ ]:
# 6.2 Informe de clasificación y matrices de confusión normalizadas
modelos_cm = [NOMBRE_SCRATCH_FINAL, NOMBRE_PRE_FINAL]
for nombre in modelos_cm:
    print(f"\n===== {nombre} =====")
    print(classification_report(y_test_ref, preds_test[nombre], labels=list(range(NUM_CLASSES)),
                                target_names=class_names, digits=3, zero_division=0))

# Una figura por modelo (15x15 celdas anotadas: legibles a tamaño de página)
matrices = {}
for nombre in modelos_cm:
    cm = confusion_matrix(y_test_ref, preds_test[nombre], labels=range(NUM_CLASSES))
    matrices[nombre] = cm
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)   # cada fila suma 1 (recall en la diagonal)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax, cbar=True,
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 8})
    ax.set_title(f"Matriz de confusión normalizada por filas (test) — {nombre}", weight="bold")
    ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()

# Comparación del F1 por clase entre ambos modelos
f1_clase = pd.DataFrame({n: f1_score(y_test_ref, preds_test[n], average=None, labels=list(range(NUM_CLASSES)),
                                     zero_division=0)
                         for n in modelos_cm}, index=class_names)
ax = f1_clase.plot.bar(figsize=(16, 5), color=["tab:blue", "tab:green"])
ax.set_title("F1-score por clase en test", weight="bold"); ax.set_ylabel("F1"); ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", alpha=0.6); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

[PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* clases mejor y peor reconocidas por cada modelo; pares de clases que más se confunden y su plausibilidad visual (p. ej. acciones que comparten postura o contexto); clases en las que el modelo preentrenado aporta más mejora respecto al *from scratch*.

## 6.3 Comparación de resultados

La tabla final reúne, para cada estrategia, el rendimiento en test, la precisión de validación, el coste (parámetros y tiempo de entrenamiento) y un indicador de generalización: la **diferencia entre la precisión de entrenamiento y la de validación en la época cuyos pesos se evalúan** (*gap*): la de mayor `val_accuracy` para la CNN *from scratch* (`ModelCheckpoint`) y la de menor `val_loss` para los preentrenados (`EarlyStopping` con `restore_best_weights`). La columna de validación corresponde también a esos mismos pesos, y `val − test` mide la coherencia entre validación y test. Un *gap* grande indica sobreajuste; precisiones bajas en ambas particiones indicarían subajuste. En los modelos con *dropout* y *data augmentation* la precisión de entrenamiento se mide con ambas técnicas activas, por lo que el *gap* puede ser pequeño o incluso negativo.

In [ ]:
# 6.3 Tabla comparativa final y visualizaciones
df_reg = funciones.tabla_resultados(resultados)

def indicadores_validacion(nombre):
    """Validación, gap train-val y coste de los PESOS EVALUADOS en test (época retenida)."""
    if nombre == NOMBRE_SCRATCH_FINAL:
        # Checkpoint de mayor val_accuracy (comprobado al recargarlo en la sección 4.5)
        fila = df_reg[(df_reg["estrategia"] == "From scratch") & (df_reg["experimento"] == nombre)].iloc[0]
        return {"val_accuracy": float(fila["best_val_accuracy"]), "gap_train_val": float(fila["gap_acc"]),
                "época retenida": int(fila["best_epoch"]), "épocas totales": int(fila["epochs_entrenadas"]),
                "tiempo_entrenamiento_min": float(fila["tiempo_s"]) / 60}
    # Preentrenados: EarlyStopping restaura la época de menor val_loss de la última fase
    h = historiales[nombre]                       # historial de la última fase (TL o FT)
    n_total = len(info_pre[nombre]["historial_total"]["loss"])
    e = int(np.argmin(h["val_loss"]))
    return {"val_accuracy": info_pre[nombre]["val_accuracy"],   # reevaluada en 5.4 con los pesos retenidos
            "gap_train_val": float(h["accuracy"][e] - h["val_accuracy"][e]),
            "época retenida": n_total - len(h["loss"]) + e + 1, "épocas totales": n_total,
            "tiempo_entrenamiento_min": info_pre[nombre]["tiempo_total_s"] / 60}

hist_total = {NOMBRE_SCRATCH_FINAL: historiales[NOMBRE_SCRATCH_FINAL]}
hist_total.update({n: info_pre[n]["historial_total"] for n in mejor_por_arq.values()})

filas_final = []
for r in eval_test:
    if r["estrategia"] == "Trivial":
        continue
    ind = indicadores_validacion(r["modelo"])
    filas_final.append({**r, **ind, "val − test": ind["val_accuracy"] - r["test_accuracy"]})
df_final = pd.DataFrame(filas_final)
display(df_final.round(4))

# Barras agrupadas: accuracy y F1-macro en test y accuracy de validación (con la referencia aleatoria)
metricas = ["test_accuracy", "test_macro_f1", "val_accuracy"]
pos = np.arange(len(df_final)); ancho = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(metricas):
    barras = ax.bar(pos + (i - 1) * ancho, df_final[m], ancho, label=m)
    ax.bar_label(barras, fmt="%.3f", fontsize=8)
ax.axhline(1 / NUM_CLASSES, color="red", linestyle="--", label="Aleatorio (1/15)")
ax.set_xticks(pos); ax.set_xticklabels(df_final["modelo"]); ax.set_ylim(0, 1)
ax.set_title("Comparación global de estrategias", weight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.6); ax.legend()
plt.tight_layout(); plt.show()

# Curvas de validación superpuestas (en la numeración de épocas propia de cada modelo)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for nombre, h in hist_total.items():
    if h is None:
        continue
    ep = np.arange(1, len(h["val_loss"]) + 1)
    ax1.plot(ep, h["val_accuracy"], "o-", label=nombre)
    ax2.plot(ep, h["val_loss"], "o-", label=nombre)
for nombre in mejor_por_arq.values():
    if "-FT" in nombre:
        ax1.axvline(len(info_pre[nombre.split("-FT")[0] + "-TL"]["historial_total"]["loss"]) + 0.5,
                    color="gray", linestyle=":", alpha=0.6)
ax1.set_title("Precisión de validación", weight="bold"); ax2.set_title("Pérdida de validación", weight="bold")
for ax in (ax1, ax2):
    ax.set_xlabel("Época"); ax.grid(True, linestyle="--", alpha=0.6); ax.legend()
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout(); plt.show()

**Análisis:** [PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis (cubrir todos los puntos exigidos):*
- **Rendimiento sobre test:** diferencia en precisión y F1-macro entre la mejor CNN *from scratch* y el mejor modelo preentrenado, y frente a los clasificadores triviales.
- **Evolución del entrenamiento y validación:** velocidad de convergencia (épocas hasta la mejor validación) y estabilidad de las curvas superpuestas.
- **Capacidad de generalización:** *gap* entrenamiento–validación y coherencia validación–test de cada estrategia.
- **Sobreajuste o subajuste:** evidencias en las curvas de cada estrategia y efecto de la regularización aplicada.
- **Tiempo/coste:** parámetros totales/entrenables y minutos de entrenamiento medidos.
- **Ventajas e inconvenientes:** *from scratch* (control total, modelo ligero y entrada pequeña, pero datos insuficientes para aprender rasgos generales) frente a preentrenados (conocimiento previo de ImageNet, mejor rendimiento con pocos datos, pero mayor tamaño, resolución 224 y dependencia de pesos externos).

## 6.4 Errores y ejemplos de predicción

Se muestran ejemplos del test interno correctamente e incorrectamente clasificados por el modelo seleccionado (`modelo_pre_final`), junto con los cinco pares de clases (real → predicha) más confundidos según su matriz de confusión, para identificar patrones de error.

In [ ]:
# 6.4 Ejemplos de aciertos y errores del mejor modelo y pares más confundidos
pred_final = preds_test[NOMBRE_PRE_FINAL]
rutas_test = rutas_all[idx_test]
aciertos = np.where(pred_final == y_test_ref)[0]
errores = np.where(pred_final != y_test_ref)[0]
n_mostrar = 8
sel_ok = rng.choice(aciertos, size=min(n_mostrar, len(aciertos)), replace=False)
sel_err = rng.choice(errores, size=min(n_mostrar, len(errores)), replace=False)

# Rejilla 4x4: dos filas de aciertos (verde) y dos de errores (rojo)
n_col = 4
fig, axes = plt.subplots(2 * n_mostrar // n_col, n_col, figsize=(11, 12.5))
for bloque, (seleccion, etiqueta, color) in enumerate([(sel_ok, "Correcta", "green"), (sel_err, "Error", "red")]):
    for k in range(n_mostrar):
        ax = axes.ravel()[bloque * n_mostrar + k]; ax.axis("off")
        if k >= len(seleccion):
            continue
        i = seleccion[k]
        img = cv2.cvtColor(cv2.imread(rutas_test[i]), cv2.COLOR_BGR2RGB)
        ax.imshow(cv2.resize(img, IMG_SIZE_PRE))
        ax.set_title(f"{etiqueta}\nReal: {class_names[y_test_ref[i]]}\nPred: {class_names[pred_final[i]]}",
                     fontsize=9, color=color)
plt.suptitle(f"Ejemplos de predicción en test — {NOMBRE_PRE_FINAL}", weight="bold")
plt.tight_layout(); plt.show()

# Top-5 pares de clases más confundidos (fuera de la diagonal)
cm_final = matrices[NOMBRE_PRE_FINAL].copy()
np.fill_diagonal(cm_final, 0)
pares = [(class_names[i], class_names[j], int(cm_final[i, j]), cm_final[i, j] / matrices[NOMBRE_PRE_FINAL][i].sum())
         for i, j in zip(*np.unravel_index(np.argsort(cm_final, axis=None)[::-1][:5], cm_final.shape))]
display(pd.DataFrame(pares, columns=["clase real", "clase predicha", "nº errores", "% de la clase real"]).round(3))
print(f"Errores totales del modelo {NOMBRE_PRE_FINAL}: {len(errores)} de {len(y_test_ref)} imágenes de test")

[PENDIENTE: completar con las métricas de la ejecución]

*Guía de análisis:* patrones comunes en los errores (acciones con postura similar, objeto clave pequeño u ocluido —teléfono, auriculares, comida—, varias personas en escena, encuadre o iluminación poco habituales); si los pares más confundidos son semánticamente cercanos; posibles errores o ambigüedades de etiquetado observados.

# 7. Conclusiones

## 7.1 Mejor modelo y estrategia

[PENDIENTE: completar con las métricas de la ejecución]

*Debe argumentarse:*
- Qué modelo y estrategia ofrecen la mejor solución, citando su precisión y F1-macro en test (sección 6.1) y su margen sobre el clasificador aleatorio.
- Por qué es preferible, no solo por la métrica: generalización (*gap*, coherencia validación–test), estabilidad del entrenamiento y coste.
- Si la diferencia frente a la otra arquitectura preentrenada compensa su coste (parámetros, tiempo).

## 7.2 Aprendizajes y análisis crítico

[PENDIENTE: completar con las métricas de la ejecución]

*Debe argumentarse:*
- Decisiones con mayor impacto en cada estrategia (p. ej. regularización y *data augmentation* en *from scratch*; *transfer learning* frente a *fine-tuning* y profundidad de descongelado en los preentrenados).
- Técnicas que ayudaron y técnicas que no aportaron mejora, con la evidencia experimental correspondiente.
- Limitaciones: tamaño del dataset, resolución distinta entre estrategias (128 frente a 224), test interno de 15 % (≈1890 imágenes), falta de validación cruzada, no determinismo de la GPU y presupuesto de cómputo.

## 7.3 Posibles mejoras futuras

[PENDIENTE: completar con las métricas de la ejecución]

*Posibles líneas a valorar según los resultados:* arquitecturas mayores (EfficientNetB3/V2, ConvNeXt, Vision Transformers); *augmentation* más fuerte (MixUp, CutMix, RandAugment); descongelado progresivo por etapas con tasas de aprendizaje discriminativas; *label smoothing*; *ensembles* de modelos; validación cruzada; uso de información temporal o de pose si se dispusiera de vídeo.

# 8. Referencias

- Nagadia, M. (s. f.). *Human Action Recognition (HAR) Dataset* [conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/meetnagadia/human-action-recognition-har-dataset
- Keras Team. *Keras Applications*. https://keras.io/api/applications/
- TensorFlow. *Transfer learning and fine-tuning* (tutorial). https://www.tensorflow.org/tutorials/images/transfer_learning
- Keras Team (Chollet, F.). *Transfer learning & fine-tuning* (guía). https://keras.io/guides/transfer_learning/
- TensorFlow. *tf.data: Build TensorFlow input pipelines*. https://www.tensorflow.org/guide/data
- TensorFlow. *Data augmentation* (tutorial). https://www.tensorflow.org/tutorials/images/data_augmentation
- Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks. *Proceedings of the 36th International Conference on Machine Learning (ICML)*, PMLR 97, 6105–6114. https://arxiv.org/abs/1905.11946
- He, K., Zhang, X., Ren, S., & Sun, J. (2016a). Deep Residual Learning for Image Recognition. *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)*, 770–778. https://arxiv.org/abs/1512.03385
- He, K., Zhang, X., Ren, S., & Sun, J. (2016b). Identity Mappings in Deep Residual Networks. *European Conference on Computer Vision (ECCV)*, LNCS 9908, 630–645. https://arxiv.org/abs/1603.05027
- Yosinski, J., Clune, J., Bengio, Y., & Lipson, H. (2014). How transferable are features in deep neural networks? *Advances in Neural Information Processing Systems 27 (NeurIPS)*, 3320–3328. https://arxiv.org/abs/1411.1792
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: A Simple Way to Prevent Neural Networks from Overfitting. *Journal of Machine Learning Research*, 15(56), 1929–1958.
- Ioffe, S., & Szegedy, C. (2015). Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift. *Proceedings of the 32nd ICML*, PMLR 37, 448–456. https://arxiv.org/abs/1502.03167
- Kingma, D. P., & Ba, J. (2015). Adam: A Method for Stochastic Optimization. *3rd International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/1412.6980
- Deng, J., Dong, W., Socher, R., Li, L.-J., Li, K., & Fei-Fei, L. (2009). ImageNet: A large-scale hierarchical image database. *IEEE CVPR*, 248–255.
- Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.

# 9. Lista de comprobación y exportación a PDF

Antes de entregar:

- [ ] Todas las secciones están completadas.
- [ ] El código se ha ejecutado y no quedan errores.
- [ ] Se muestran evidencias de entrenamiento real para todos los modelos presentados.
- [ ] Las métricas y figuras proceden de la ejecución del notebook.
- [ ] El conjunto de test se ha reservado para la evaluación final.
- [ ] Las conclusiones están respaldadas por los experimentos.
- [ ] El PDF incluye explicaciones, código ejecutado, resultados y figuras.
- [ ] `QUICK_RUN = False` en la ejecución entregada.

**Exportación:** el PDF debe generarse a partir del `.ipynb` **ya ejecutado y guardado**, para que incluya celdas, salidas y figuras. La celda siguiente funciona en Colab y en local (desactivada por defecto con `EXPORTAR_PDF = False`):

- `--to webpdf` (recomendado): renderiza el notebook con Chromium; requiere `pip install "nbconvert[webpdf]"` y descarga Chromium (`--allow-chromium-download` o `playwright install chromium`). No necesita LaTeX.
- `--to pdf`: vía LaTeX; requiere `pandoc` y XeLaTeX (en Colab: paquetes `texlive-xetex`, `texlive-fonts-recommended`, `texlive-plain-generic`; en macOS: MacTeX). Es la opción de respaldo.

En Colab, el notebook debe estar guardado en Google Drive (Archivo → Guardar) y se monta Drive para acceder al fichero.

In [ ]:
# 9. Exportación a PDF (Colab o local). No se ejecuta salvo que EXPORTAR_PDF = True.
import subprocess, sys

EXPORTAR_PDF = False
NOMBRE_NOTEBOOK = "Plantilla_Proyecto_07MIAR.ipynb"
FORMATO = "webpdf"   # "webpdf" (Chromium, sin LaTeX) o "pdf" (LaTeX)

def ejecutar(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if EXPORTAR_PDF:
    try:
        import google.colab  # noqa: F401
        EN_COLAB = True
    except ImportError:
        EN_COLAB = False

    if EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        ruta_nb = f"/content/drive/MyDrive/Colab Notebooks/{NOMBRE_NOTEBOOK}"
        if FORMATO == "pdf":
            ejecutar(["apt-get", "-qq", "install", "-y", "pandoc", "texlive-xetex",
                      "texlive-fonts-recommended", "texlive-plain-generic"])
    else:
        ruta_nb = os.path.abspath(NOMBRE_NOTEBOOK)

    if FORMATO == "webpdf":
        ejecutar([sys.executable, "-m", "pip", "install", "-q", "nbconvert[webpdf]"])
        ejecutar([sys.executable, "-m", "jupyter", "nbconvert", "--to", "webpdf",
                  "--allow-chromium-download", ruta_nb])
    else:
        ejecutar([sys.executable, "-m", "jupyter", "nbconvert", "--to", "pdf", ruta_nb])
    print("PDF generado junto a:", ruta_nb)
else:
    print("Exportación desactivada (EXPORTAR_PDF = False).")